In [2]:
# CONSTANTS.py
EMOTIONS_LIST = ("행복/만족", "우울/피곤", "불안/걱정", "분노/짜증", "설렘/흥분", "외로움/공허", "평온/안정")

EMOTION_LEXICON = {
    "행복/만족": ["행복", "기쁘", "즐거", "좋다", "만족"],
    "우울/피곤": ["우울", "슬프", "피곤", "지치", "힘들다"],
    "불안/걱정": ["불안", "걱정", "초조", "두렵", "무섭", "어떡"],
    "분노/짜증": ["화나", "짜증", "분노", "열받"],
    "설렘/흥분": ["설렘", "기대", "흥분", "두근"],
    "외로움/공허": ["외롭", "공허", "허전"],
    "평온/안정": ["편안", "안정", "차분", "평온", "다행"]
}

In [3]:
!git clone https://github.com/LeegoYuz/2026_OSSProgramming_42

fatal: destination path '2026_OSSProgramming_42' already exists and is not an empty directory.


In [4]:
%cd /content/2026_OSSProgramming_42
!git checkout 박지우/시스템연결
!git pull origin 박지우/시스템연결

/content/2026_OSSProgramming_42
D	README.md
Already on '박지우/시스템연결'
Your branch is up to date with 'origin/박지우/시스템연결'.
From https://github.com/LeegoYuz/2026_OSSProgramming_42
 * branch            박지우/시스템연결 -> FETCH_HEAD
Already up to date.


In [5]:
%cd /content/2026_OSSProgramming_42/0_프로젝트파일/project

/content/2026_OSSProgramming_42/0_프로젝트파일/project


In [6]:
!pip install -U transformers datasets evaluate accelerate

  Using cached transformers-5.12.1-py3-none-any.whl.metadata (33 kB)
Using cached transformers-5.12.1-py3-none-any.whl (11.2 MB)


In [1]:
import torch
import numpy as np
from datasets import load_dataset
from transformers import ElectraTokenizer, ElectraForSequenceClassification, TrainingArguments, Trainer
import evaluate

In [7]:
# 1. 감정 라벨 정의
EMOTIONS_LIST = ["행복/만족", "우울/피곤", "불안/걱정", "분노/짜증", "설렘/흥분", "외로움/공허", "평온/안정"]

# 2. 데이터셋 로드 (CSV 예시: text,label)
dataset = load_dataset(
    "csv",
    data_files={
        "train": "/content/2026_OSSProgramming_42/0_프로젝트파일/project/dataset/train.csv",
        "validation": "/content/2026_OSSProgramming_42/0_프로젝트파일/project/dataset/valid.csv"
    }
)

In [8]:
# 3. 토크나이저
MODEL_NAME = "monologg/koelectra-base-v3-discriminator"
tokenizer = ElectraTokenizer.from_pretrained(MODEL_NAME)

def tokenize(batch):
    return tokenizer(batch["text"], truncation=True, padding="max_length", max_length=128)

dataset = dataset.map(tokenize, batched=True)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [9]:
# 4. 모델 생성 (num_labels=7)
model = ElectraForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=len(EMOTIONS_LIST))

# 5. 메트릭 정의
accuracy = evaluate.load("accuracy")
f1 = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    acc = accuracy.compute(predictions=preds, references=labels)["accuracy"]
    f1_macro = f1.compute(predictions=preds, references=labels, average="macro")["f1"]
    return {"accuracy": acc, "f1_macro": f1_macro}

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] ElectraForSequenceClassification LOAD REPORT from: monologg/koelectra-base-v3-discriminator
Key                                               | Status     | 
--------------------------------------------------+------------+-
discriminator_predictions.dense_prediction.bias   | UNEXPECTED | 
discriminator_predictions.dense_prediction.weight | UNEXPECTED | 
discriminator_predictions.dense.weight            | UNEXPECTED | 
discriminator_predictions.dense.bias              | UNEXPECTED | 
classifier.out_proj.weight                        | MISSING    | 
classifier.out_proj.bias                          | MISSING    | 
classifier.dense.weight                           | MISSING    | 
classifier.dense.bias                             | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your do

In [11]:
# 6. 학습 설정
training_args = TrainingArguments(
    output_dir="./results",
    do_eval=True,              # 평가 활성화
    logging_dir="./logs",      # 로그 저장
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01
)

[transformers] `logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [13]:
# 7. Trainer 실행
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
    compute_metrics=compute_metrics
)

trainer.train()

Step,Training Loss


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=24, training_loss=1.932461420694987, metrics={'train_runtime': 21.1244, 'train_samples_per_second': 16.048, 'train_steps_per_second': 1.136, 'total_flos': 22299662995200.0, 'train_loss': 1.932461420694987, 'epoch': 3.0})

In [14]:
# 8. 모델 저장
trainer.save_model("./model")
tokenizer.save_pretrained("./model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./model/tokenizer_config.json', './model/tokenizer.json')